In [ ]:
# pip install pandas

import pandas as pd

# Büyük veri dosyasını parça parça oku
chunks = pd.read_csv("final_combined_data.csv", chunksize=5000, low_memory=False)

filtered_data = []

for chunk in chunks:
    # Temizleme
    chunk = chunk.replace({'NA': None, 'Unknown': None, 'Not Reported': None})
    chunk['TMB'] = pd.to_numeric(chunk['TMB'], errors='coerce')
    chunk['PD-L1'] = pd.to_numeric(chunk['Clinically reported PD-L1 score'], errors='coerce')

    # Klinik eşiğe göre filtrele (TMB ≥ 3, PD-L1 ≥ 1)
    filtered = chunk[(chunk['TMB'] >= 3) & (chunk['PD-L1'] >= 1)]

    # Gerekli sütunlar eksikse at
    filtered = filtered.dropna(subset=['MSI Type', 'Durable clinical response', 'TMB', 'PD-L1'])

    # Uygun olan satırları listeye ekle
    filtered_data.append(filtered)

# Hepsini birleştir
final_df = pd.concat(filtered_data, ignore_index=True)

# Yeni düzenlenmiş veri setini kaydet
final_df.to_csv("final_filtered_clinical_dataset.csv", index=False)

print("Yeni dosya başarıyla kaydedildi: final_filtered_clinical_dataset.csv")


In [ ]:
# pip install pandas numpy matplotlib scikit-learn shap scikit-fuzzy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
import shap
import skfuzzy as fuzz
df = pd.read_csv("data/final_filtered_clinical_dataset.csv")

features = ['Clinically reported PD-L1 score', 'TMB', 'MSI Type']
target = 'Durable clinical response'

df = df.replace({'NA': np.nan, 'Unknown': np.nan, 'Not Reported': np.nan})
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df = df.dropna(subset=features + [target])

le = LabelEncoder()
df['MSI Type'] = le.fit_transform(df['MSI Type'])
df[target] = le.fit_transform(df[target])
X = df[features]
y = df[target]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("\n🔍 Classification Report:\n")
print(classification_report(y_test, y_pred))
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test[:100], check_additivity=False)

# Genel bar grafiği
if isinstance(shap_values.values, list) or shap_values.values.ndim == 3:
    values = shap_values.values[:, :, 1]
else:
    values = shap_values.values

mean_abs_shap = np.abs(values).mean(axis=0)
feature_names = X.columns.tolist()
top_indices = np.argsort(mean_abs_shap)[-10:][::-1]
top_features = [feature_names[i] for i in top_indices]
top_values = mean_abs_shap[top_indices]

plt.figure(figsize=(8, 5))
plt.barh(top_features[::-1], top_values[::-1])
plt.xlabel("Ortalama |SHAP| Değeri")
plt.title("SHAP Özellik Önem Grafiği (Genel)")
plt.tight_layout()
plt.show()
hasta_indeksi = 1 # ← buradan hastayı seçebilirsin
print(f"\n🎯 Hasta indeksi: {hasta_indeksi}")
shap_values.feature_names = X.columns.tolist()
shap.plots.waterfall(shap_values[hasta_indeksi, :, 1])
pdl1 = df.iloc[hasta_indeksi]['Clinically reported PD-L1 score']
tmb = df.iloc[hasta_indeksi]['TMB']

x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)

pdl1_low = fuzz.trapmf(x_pdl1, [0, 0, 10, 30])
pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_low = fuzz.trapmf(x_tmb, [0, 0, 5, 10])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

pdl1_l = fuzz.interp_membership(x_pdl1, pdl1_low, pdl1)
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_l = fuzz.interp_membership(x_tmb, tmb_low, tmb)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)

if tmb_h > 0.5:
    if pdl1_h > 0.3:
        suggestion = "🟢 Monoterapi (PD-1 inhibitörü) önerilir."
    else:
        suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir."
else:
    suggestion = "🔴 Daha fazla biyobelirteç verisi gerekli."

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}")
print("Öneri:", suggestion)


In [ ]:
# pip install scikit-fuzzy
import pandas as pd
import numpy as np
import skfuzzy as fuzz

# Veri setini yükle
df = pd.read_csv("final_combined_data.csv")
df = df.replace({'NA': None, 'Unknown': None, 'Not Reported': None})
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df['PD-L1'] = pd.to_numeric(df['Clinically reported PD-L1 score'], errors='coerce')
df = df.dropna(subset=['TMB', 'PD-L1'])

# Bulanık kümeleri tanımla
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

print("\n🟡 Kombinasyon önerisi verilen hastaların indeksleri:")

for i in range(len(df)):
    pdl1 = df.iloc[i]['Clinically reported PD-L1 score']
    tmb = df.iloc[i]['TMB']

    pdl1_l = fuzz.interp_membership(x_pdl1, pdl1_low, pdl1)
    pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
    tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)

    if tmb_h > 0.5 and pdl1_h <= 0.3:
        print(f"→ hasta_indeksi = {i} | PD-L1: {pdl1}, TMB: {tmb}")



In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, accuracy_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# 📄 VERİ YÜKLEME ve TEMİZLEME
df = pd.read_csv("final_filtered_clinical_dataset.csv")
df['Durable clinical response'] = df['Durable clinical response'].replace({
    "Durable clinical benefit": 1,
    "No durable clinical benefit": 0
})
df = df[df['Durable clinical response'].isin([0, 1])].copy()

# 📌 ÖZELLİKLER
features = [
    'Clinically reported PD-L1 score', 'PD-L1', 'TMB', 'MSI Type',
    'Tumor Purity', 'Mutation Count', 'Fraction Genome Altered', 'Cancer Type Detailed'
]
target = 'Durable clinical response'

# 🔧 DÖNÜŞÜMLER
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df = df.dropna(subset=features + [target])
df['MSI Type'] = LabelEncoder().fit_transform(df['MSI Type'])
df[target] = df[target].astype(int)

X = df[features]
y = df[target]

# 🔨 ÖN İŞLEME
num_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer()), ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("num", num_pipe, num_features),
                                  ("cat", cat_pipe, cat_features)])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🧪 TRAIN/TEST BÖLÜNMESİ
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# ⚖️ SADECE EĞİTİM VERİSİNE SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# 🔧 scale_pos_weight
neg = sum(y_train_resampled == 0)
pos = sum(y_train_resampled == 1)
scale_pos_weight = neg / pos

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                      scale_pos_weight=scale_pos_weight, random_state=42)
model.fit(X_train_resampled, y_train_resampled)
y_proba = model.predict_proba(X_test)[:, 1]
proba_corrected = 1 - y_proba  # class_1 yerine class_0 çıktığı için tersliyoruz
y_pred = model.predict(X_test)

# 🔍 EŞİK = 0.5 DEĞERLENDİRME
print("\n🔍 Classification Report (Threshold = 0.5):\n")
print(classification_report(y_test, y_pred))
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("📈 ROC AUC Score (terslenmiş):", roc_auc_score(y_test, proba_corrected))

# 🎯 OPTİMUM EŞİK BULMA
prec, recall, thresholds = precision_recall_curve(y_test, proba_corrected)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
print(f"\n📍 Optimum Threshold: {best_thresh:.2f}")
print(f"📈 En yüksek F1-score: {np.max(f1_scores):.3f}")

# 🧪 YENİ EŞİK İLE TAHMİN
y_pred_opt = (proba_corrected > best_thresh).astype(int)

# 🔍 OPTİMUM EŞİK DEĞERLENDİRME
print("\n🔍 Classification Report (Optimum Eşik ile):\n")
print(classification_report(y_test, y_pred_opt))
print("✅ Accuracy (optimum eşik):", accuracy_score(y_test, y_pred_opt))
print("📈 ROC AUC Score (doğru):", roc_auc_score(y_test, proba_corrected))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt)
plt.title("Optimum Eşik ile Confusion Matrix")
plt.show()

# 🧠 SHAP
explainer = shap.Explainer(model, X_train_resampled)
X_test_df = pd.DataFrame(X_test[:50], columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)
if isinstance(shap_values.values, list) or shap_values.values.ndim == 3:
    shap_values.values = shap_values.values[:, :, 1]

shap.plots.bar(shap_values)
shap.plots.waterfall(shap_values[0])

# 🤖 BULANIK MANTIK TEDAVİ ÖNERİSİ
hasta_indeksi = 6
pdl1 = df.iloc[hasta_indeksi]['Clinically reported PD-L1 score']
tmb = df.iloc[hasta_indeksi]['TMB']

x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)

if tmb_h > 0.5:
    if pdl1_h > 0.3:
        suggestion = "🟢 Monoterapi (PD-1 inhibitörü) önerilir."
    else:
        suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir."
else:
    suggestion = "🔴 Daha fazla biyobelirteç verisi gerekli."

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}")
print("Öneri:", suggestion)
print(classification_report(y_test, y_pred_opt))



In [ ]:
import numpy as np
import pandas as pd
import skfuzzy as fuzz

# PD-L1 ve TMB bulanık kümeleri
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

# Kombinasyon önerisi verilen hastaların indekslerini topla
kombinasyon_indices = []

for i in df.index:
    try:
        pdl1 = df.loc[i]['Clinically reported PD-L1 score']
        tmb = df.loc[i]['TMB']
        if pd.notnull(pdl1) and pd.notnull(tmb):
            pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
            tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
            if tmb_h > 0.5 and pdl1_h <= 0.3:
                kombinasyon_indices.append(i)
    except:
        continue

print("🔹 Kombinasyon tedavisi önerilen hasta indeksleri:")
print(kombinasyon_indices)
